# NIFTY ML TICK COLLECTOR — v4 (Perfect Edition)
**Databricks Community / Jobs · `^NSEI` · 1-min bars · IST-aware**

**Delay ->60-90seconds**

| Layer | Detail |
|---|---|
| Fetch logic | `period="1d", interval="1m"` — proven stable for `^NSEI` |
| Bar selection | `iloc[-2]` = last **fully-closed** bar; `iloc[-1]` is the live partial bar |
| Timestamp | `bar_ts` from `raw.index[-2]` (IST-converted) — no wall-clock mismatch |
| Market-close detection | 5 consecutive bad ticks triggers graceful abort |
| Stale-bar dedup | `bar_ts == last_bar_ts` guard prevents duplicate rows |
| Collapsed-bar guard | `H=L=O=C` check catches yfinance partial-bar artefact |
| VWAP | Running intraday cumulative TP×max(V,1) — handles index 0-volume |
| Self-tests | 50+ assertions across every helper — runs before first tick |
| Storage | Delta Lake + CSV + Parquet (three redundant sinks) |
| Schedule | `0 3 * * 1-5` UTC = 08:30 IST (45-min warm-up) |


In [0]:
%pip install yfinance pyarrow --quiet


## 1 · Imports & Configuration
All config is **hardcoded** (no widgets) — safe for Databricks Jobs where
`dbutils.widgets` cannot be read during scheduled runs.
Change values here only; never inline magic numbers elsewhere.


In [0]:
# ── Imports ───────────────────────────────────────────────────────────────────
import datetime
import math
import os
import time
import warnings
import zoneinfo
from calendar import monthrange
from typing import Optional, Tuple

import pandas as pd
import yfinance as yf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

warnings.filterwarnings("ignore")

# ── Timezone & Spark ──────────────────────────────────────────────────────────
IST   = zoneinfo.ZoneInfo("Asia/Kolkata")
spark = SparkSession.builder.getOrCreate()

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION — edit only this block
# ══════════════════════════════════════════════════════════════════════════════
SYMBOL         : str   = "^NSEI"          # Yahoo Finance ticker — do NOT change
INTERVAL_MIN   : int   = 1                # Bar interval in minutes (1–60)
INTERVAL_SEC   : int   = INTERVAL_MIN * 60

# NSE weekly expiry weekday: 0=Mon 1=Tue 2=Wed 3=Thu 4=Fri
# Nifty 50 weekly expiry migrated to TUESDAY effective Apr 4 2025
EXPIRY_WEEKDAY : int   = 1                # 1 = Tuesday

BASE_PATH      : str   = "/Workspace/Users/hada.jai.hind@gmail.com/project_MarketMinds/Nifty_data_collection"
DELTA_TABLE    : str   = "nifty.ticks"

# Flush buffer to storage every N ticks (also flushed at graceful exit)
FLUSH_EVERY    : int   = 10

# Abort after this many consecutive fetch-failures (market-close proxy)
MAX_BAD_TICKS  : int   = 5

# Single-bar move above this fraction is labelled SPIKE (not filtered out)
SPIKE_PCT      : float = 0.05             # 5 %

# ── Derived paths ─────────────────────────────────────────────────────────────
CSV_DIR        : str   = f"{BASE_PATH}/csv"
PARQUET_DIR    : str   = f"{BASE_PATH}/parquet"

# NSE trading window in total-minutes-since-midnight (IST)
_T_OPEN        : int   = 9  * 60 + 15    # 09:15
_T_CLOSE       : int   = 15 * 60 + 33    # 15:30 to get comeplete data rows

# Spark infers Python int → LongType; cast these columns back to INT
_INT_COLS = [
    "year", "month", "date", "day", "hour", "minute",
    "week_of_month", "session_number",
    "expiry", "is_monthly_expiry", "days_to_expiry",
]

_WD = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri"}
print("=" * 64)
print(f"  NIFTY ML COLLECTOR  v4  |  {SYMBOL}  |  {INTERVAL_MIN}-min bars")
print(f"  EXPIRY_WEEKDAY = {EXPIRY_WEEKDAY} ({_WD[EXPIRY_WEEKDAY]})")
print(f"  Delta table    = {DELTA_TABLE}")
print(f"  Flush every    = {FLUSH_EVERY} ticks   |  Bad-tick limit = {MAX_BAD_TICKS}")
print(f"  Spike guard    = {SPIKE_PCT*100:.0f}%")
print("=" * 64)


## 2 · Config Validation

In [0]:
def _validate_config() -> None:
    """
    Eagerly validates every config constant.
    Raises ValueError with ALL errors listed (not just the first).
    """
    errors = []
    if not SYMBOL:
        errors.append("SYMBOL must be non-empty")
    if not (0 <= EXPIRY_WEEKDAY <= 4):
        errors.append(f"EXPIRY_WEEKDAY must be 0-4, got {EXPIRY_WEEKDAY}")
    if not (1 <= INTERVAL_MIN <= 60):
        errors.append(f"INTERVAL_MIN must be 1-60, got {INTERVAL_MIN}")
    if not BASE_PATH:
        errors.append("BASE_PATH must be non-empty")
    if not DELTA_TABLE:
        errors.append("DELTA_TABLE must be non-empty")
    if FLUSH_EVERY < 1:
        errors.append(f"FLUSH_EVERY must be >= 1, got {FLUSH_EVERY}")
    if MAX_BAD_TICKS < 1:
        errors.append(f"MAX_BAD_TICKS must be >= 1, got {MAX_BAD_TICKS}")
    if not (0.001 <= SPIKE_PCT <= 0.5):
        errors.append(f"SPIKE_PCT must be 0.001-0.5, got {SPIKE_PCT}")
    if errors:
        raise ValueError("CONFIG ERRORS:\n" + "\n".join(f"  · {e}" for e in errors))

_validate_config()
print("  _validate_config() ✓")


## 3 · Calendar & Session Helpers

In [0]:
def _last_expiry_of_month(year: int, month: int) -> datetime.date:
    """
    Last occurrence of EXPIRY_WEEKDAY in *month*.
    Example: last Tuesday of March 2026 = 2026-03-31
    """
    last = monthrange(year, month)[1]
    d    = datetime.date(year, month, last)
    while d.weekday() != EXPIRY_WEEKDAY:
        d -= datetime.timedelta(days=1)
    return d


def _days_to_next_expiry(d: datetime.date) -> int:
    """
    Calendar days until the next occurrence of EXPIRY_WEEKDAY.
    Returns 0 on expiry day itself.

    Formula: (target_weekday - current_weekday) % 7
      · Mon(0) to Tue(1): (1-0)%7 = 1
      · Tue(1) to Tue(1): (1-1)%7 = 0   ← expiry day
      · Wed(2) to Tue(1): (1-2)%7 = 6
    """
    return (EXPIRY_WEEKDAY - d.weekday()) % 7


def _week_of_month(d: datetime.date) -> int:
    """1-based week number within the month (day 1-7 → 1, 8-14 → 2, …)"""
    return (d.day - 1) // 7 + 1


def _session_number(h: int) -> int:
    """
    ML-friendly intraday session bucket (int 0-6):
      0 = outside market hours
      1 = 09:xx   2 = 10:xx   3 = 11:xx
      4 = 12:xx   5 = 13:xx   6 = 14-15:xx  (afternoon + close)
    """
    return {9: 1, 10: 2, 11: 3, 12: 4, 13: 5, 14: 6, 15: 6}.get(h, 0)


def _is_market_open() -> bool:
    """
    True iff current IST wall-clock is within NSE trading hours
    AND today is a weekday (Mon-Fri).
    Does NOT account for NSE holidays — add a holiday calendar if needed.
    """
    now = datetime.datetime.now(IST)
    if now.weekday() >= 5:            # Saturday=5, Sunday=6
        return False
    t = now.hour * 60 + now.minute
    return _T_OPEN <= t <= _T_CLOSE


def _market_open_dt() -> datetime.datetime:
    """Today's 09:15:00 IST as a timezone-aware datetime."""
    now = datetime.datetime.now(IST)
    return now.replace(hour=9, minute=15, second=0, microsecond=0)


def _market_close_dt() -> datetime.datetime:
    """Today's 15:30:00 IST as a timezone-aware datetime."""
    now = datetime.datetime.now(IST)
    return now.replace(hour=15, minute=33, second=0, microsecond=0)


## 4 · Data Quality Classifier

In [0]:
def _data_quality(close: float, volume: float, prev_close: Optional[float]) -> str:
    """
    Classify a completed bar's data quality.

    Labels (stored verbatim in `data_quality` column):
      BAD_PRICE  — close <= 0 (corrupt / missing data)
      SPIKE      — single-bar absolute move > SPIKE_PCT of prev_close
                   (possible data error; labelled but NOT dropped)
      OK_NO_VOL  — volume == 0, price valid
                   (^NSEI is an index — volume is always 0, this is expected)
      OK         — clean bar with real volume

    Design notes:
      · SPIKE rows are kept in the dataset with label SPIKE so the ML model
        can learn to handle thin-market artefacts; they are NOT silently dropped.
      · The `prev_close` guard means SPIKE cannot fire on the very first bar
        of a session (no reference yet) — first bar is OK or OK_NO_VOL.
    """
    if close <= 0:
        return "BAD_PRICE"
    if prev_close is not None and prev_close > 0:
        if abs(close - prev_close) / prev_close > SPIKE_PCT:
            return "SPIKE"
    if volume == 0:
        return "OK_NO_VOL"
    return "OK"


## 5 · Core Fetch Function
### Key design decisions
| Decision | Rationale |
|---|---|
| `period="1d"` | yfinance raises `ValueError` when `start`/`end` include a time-component for index tickers; `period` avoids this entirely |
| `iloc[-2]` not `iloc[-1]` | `iloc[-1]` is the **current in-progress minute** — yfinance returns it with collapsed `H=L=O=C` until the candle closes. `iloc[-2]` is the last **fully-closed** bar. |
| `bar_ts = raw.index[-2]` | Timestamp comes from the **bar itself** (IST-converted index), NOT from wall-clock — prevents off-by-one on slow networks |
| `repair=False` | Suppresses yfinance's auto-repair network call, faster & deterministic |
| `max(vo, 1)` in VWAP | Prevents VWAP accumulator from staying zero forever for index tickers |


In [0]:
def _fetch_bar(
    prev_close : Optional[float],
    cum_tp_vol : float,
    cum_vol    : float,
) -> Tuple[Optional[dict], float, float, Optional[pd.Timestamp]]:
    """
    Download the latest fully-closed 1-min bar for SYMBOL.

    Returns
    -------
    row        : dict | None — feature-rich tick row ready for storage
    cum_tp_vol : float       — updated running TP×V accumulator (for VWAP)
    cum_vol    : float       — updated running volume accumulator (for VWAP)
    bar_ts     : Timestamp | None — bar's own IST timestamp (for stale-bar dedup)

    Failure modes that return (None, cum_tp_vol, cum_vol, None_or_ts):
      · yfinance network exception
      · empty DataFrame
      · no rows after timezone-convert + dedup
      · bad close price (<= 0)
      · collapsed bar (H=L=O=C on a single-row result)
    """
    t0 = time.perf_counter()

    # ── 1. Download ───────────────────────────────────────────────────────────
    try:
        raw = yf.download(
            SYMBOL,               # "^NSEI" — ONLY positional arg; never pass tickers= here
            period      = "1d",   # DO NOT use start=/end= for ^NSEI — causes ValueError
            interval    = "1m",
            progress    = False,
            auto_adjust = True,   # adjusts for splits; makes O/H/L/C directly comparable
            repair      = False,  # skip repair network call → faster, deterministic
        )
    except Exception as exc:
        print(f"    [yfinance ERROR] {exc}")
        return None, cum_tp_vol, cum_vol, None

    latency_ms = (time.perf_counter() - t0) * 1000

    # ── 2. Empty response guard ───────────────────────────────────────────────
    if raw is None or raw.empty:
        print("    [WARN] Empty response from yfinance")
        return None, cum_tp_vol, cum_vol, None

    # ── 3. Flatten MultiIndex columns (yfinance >= 0.2 returns Price/Ticker) ──
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    raw.columns = [str(c).lower() for c in raw.columns]

    # ── 4. Convert index to IST ───────────────────────────────────────────────
    # yfinance returns UTC-aware index for intraday data; convert to IST so
    # bar_ts lines up with Indian calendar (hour, minute, date, etc.)
    if raw.index.tz is None:
        raw.index = raw.index.tz_localize("UTC").tz_convert(IST)
    else:
        raw.index = raw.index.tz_convert(IST)

    # ── 5. Drop duplicate timestamps (keep last = most up-to-date value) ──────
    raw = raw[~raw.index.duplicated(keep="last")]

    if len(raw) == 0:
        print("    [WARN] No rows after dedup")
        return None, cum_tp_vol, cum_vol, None

    # ── 6. Bar selection: iloc[-2] = last fully-closed bar ────────────────────
    #
    #  WHY iloc[-2]?
    #  yfinance always includes the CURRENT in-progress minute as iloc[-1].
    #  During that minute the bar is partial: H=L=O=C until the candle closes.
    #  This partial bar is indistinguishable from a real flat candle.
    #  Using iloc[-2] guarantees we always record a bar that yfinance has
    #  received the final tick for.
    #
    #  TIMESTAMP ALIGNMENT:
    #  bar_ts = raw.index[-2]  ← the bar's OWN IST timestamp, not wall-clock.
    #  This is critical: if the fetch call takes 3+ seconds, wall-clock ≠ bar time.
    #  Storing bar_ts instead of now() prevents timestamp drift.
    #
    #  EDGE CASE: only 1 bar in response (first minute of session, or very early
    #  pre-market fetch). Fall back to iloc[-1] with a warning — this is rare.
    #
    if len(raw) >= 2:
        bar    = raw.iloc[-2]
        bar_ts = raw.index[-2]
    else:
        # Only 1 bar returned — this is the very first bar of the day.
        # Accept it but note it may be partial.
        print("    [INFO] Only 1 bar in response — using iloc[-1] (first bar of session)")
        bar    = raw.iloc[-1]
        bar_ts = raw.index[-1]

    # ── 7. Extract OHLCV scalars ──────────────────────────────────────────────
    def _f(key: str) -> float:
        v = bar.get(key, 0)
        return float(v) if (v is not None and v == v) else 0.0   # NaN-safe

    o  = _f("open")
    hi = _f("high")
    lo = _f("low")
    cl = _f("close")
    vo = _f("volume")

    # ── 8. Collapsed-bar guard ────────────────────────────────────────────────
    # A bar where H=L=O=C is either:
    #   (a) a genuine flat tick (extremely rare for Nifty, range is always > 0)
    #   (b) a yfinance artefact when iloc[-1] slips through (defensive)
    # Treat as skip — do NOT count as bad_streak (it's a data artefact, not
    # a network failure).
    if hi == lo == o == cl and cl > 0:
        print(f"    [WARN] Collapsed bar at {bar_ts.strftime('%H:%M')} "
              f"(H=L=O=C={cl:.2f}) — skipping (not counted as bad tick)")
        return None, cum_tp_vol, cum_vol, bar_ts   # return bar_ts so stale-dedup works

    # ── 9. Bad price guard ────────────────────────────────────────────────────
    if cl <= 0:
        print(f"    [WARN] Bad close {cl} at {bar_ts} — skipping")
        return None, cum_tp_vol, cum_vol, bar_ts

    # ── 10. Running intraday VWAP ─────────────────────────────────────────────
    # ^NSEI volume is always 0 (it's an index, not traded).
    # Using max(vo, 1) prevents the accumulator from remaining at zero;
    # result is equal-weighted typical price when volume=0, which is the
    # standard fallback for index VWAP.
    tp          = (hi + lo + cl) / 3
    cum_tp_vol += tp * max(vo, 1)
    cum_vol    += max(vo, 1)
    vwap        = cum_tp_vol / cum_vol

    # ── 11. Timestamp dimensions from bar_ts (NOT wall-clock) ─────────────────
    #
    #  CRITICAL: Use bar_ts for calendar features (year, month, date, hour,
    #  minute) because bar_ts is the bar's own IST timestamp.
    #  wall-clock (now_ist) is used ONLY for tick_ts (the collection timestamp).
    #
    #  Example of why this matters:
    #    Bar at 10:14 IST; fetch completes at 10:15:03 IST.
    #    If we used now_ist.minute we'd store minute=15 for a 10:14 bar.
    #    bar_ts.minute correctly stores minute=14.
    #
    bar_dt = bar_ts.to_pydatetime()           # tz-aware, IST
    d      = bar_dt.date()
    h      = bar_dt.hour
    m      = bar_dt.minute
    now_ist = datetime.datetime.now(IST)      # wall-clock for tick_ts only

    # ── 12. Expiry features ───────────────────────────────────────────────────
    is_exp     = 1 if d.weekday() == EXPIRY_WEEKDAY else 0
    days_to    = _days_to_next_expiry(d)
    last_exp   = _last_expiry_of_month(d.year, d.month)
    is_monthly = 1 if (d.weekday() == EXPIRY_WEEKDAY and d == last_exp) else 0

    # ── 13. Price-derived ML features ─────────────────────────────────────────
    price_range  = round(hi - lo, 2)
    return_pct   = round((cl - o) / o * 100, 4)              if o           > 0 else 0.0
    body_pct     = round(abs(cl - o) / price_range * 100, 2) if price_range > 0 else 0.0
    upper_shadow = round(hi - max(o, cl), 2)
    lower_shadow = round(min(o, cl) - lo, 2)
    hl_ratio     = round(hi / lo, 6)                         if lo          > 0 else 1.0
    log_ret      = round(math.log(cl / prev_close), 6) \
                   if (prev_close is not None and prev_close > 0 and cl > 0) else 0.0

    row = {
        # ── Temporal ─────────────────────────────────────────────────────────
        # tick_ts   : wall-clock moment the row was collected (naive for Spark)
        # bar_time  : the bar's own IST time string — used for display + joins
        "tick_ts"           : now_ist.replace(tzinfo=None),
        "bar_ts"            : bar_dt.replace(tzinfo=None),          # bar's own time (naive)
        "tick_time_str"     : now_ist.strftime("%H:%M"),
        "bar_time_str"      : bar_dt.strftime("%H:%M"),             # bar's time string
        "symbol"            : SYMBOL,

        # ── Calendar dimensions (from bar_ts, NOT wall-clock) ────────────────
        "year"              : d.year,
        "month"             : d.month,
        "date"              : d.day,
        "day"               : d.weekday(),                          # 0=Mon … 4=Fri
        "hour"              : h,
        "minute"            : m,
        "week_of_month"     : _week_of_month(d),

        # ── OHLCV ────────────────────────────────────────────────────────────
        "open"              : round(o,  2),
        "high"              : round(hi, 2),
        "low"               : round(lo, 2),
        "close"             : round(cl, 2),
        "volume"            : round(vo, 0),

        # ── Core ML features ─────────────────────────────────────────────────
        "typical_price"     : round(tp,           2),
        "vwap"              : round(vwap,          2),
        "price_range"       : price_range,
        "return_pct"        : return_pct,
        "log_return"        : log_ret,
        "body_pct"          : body_pct,
        "upper_shadow"      : upper_shadow,
        "lower_shadow"      : lower_shadow,
        "hl_ratio"          : hl_ratio,

        # ── Session & expiry ─────────────────────────────────────────────────
        "session_number"    : _session_number(h),
        "expiry"            : is_exp,
        "is_monthly_expiry" : is_monthly,
        "days_to_expiry"    : days_to,

        # ── Ops metadata ─────────────────────────────────────────────────────
        "data_quality"      : _data_quality(cl, vo, prev_close),
        "fetch_latency_ms"  : round(latency_ms, 1),
    }
    return row, cum_tp_vol, cum_vol, bar_ts


## 6 · Storage Helpers (Delta + CSV + Parquet)

In [0]:
# ── Delta schema includes both tick_ts (wall-clock) and bar_ts (bar time) ────
_ALL_INT_COLS = [
    "year", "month", "date", "day", "hour", "minute",
    "week_of_month", "session_number",
    "expiry", "is_monthly_expiry", "days_to_expiry",
]


def _bootstrap_delta() -> None:
    """Create database and Delta table if they don't exist."""
    db = DELTA_TABLE.rsplit(".", 1)[0] if "." in DELTA_TABLE else "default"
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {DELTA_TABLE} (
            tick_ts             TIMESTAMP,
            bar_ts              TIMESTAMP,
            tick_time_str       STRING,
            bar_time_str        STRING,
            symbol              STRING,
            year                INT,
            month               INT,
            date                INT,
            day                 INT,
            hour                INT,
            minute              INT,
            week_of_month       INT,
            open                DOUBLE,
            high                DOUBLE,
            low                 DOUBLE,
            close               DOUBLE,
            volume              DOUBLE,
            typical_price       DOUBLE,
            vwap                DOUBLE,
            price_range         DOUBLE,
            return_pct          DOUBLE,
            log_return          DOUBLE,
            body_pct            DOUBLE,
            upper_shadow        DOUBLE,
            lower_shadow        DOUBLE,
            hl_ratio            DOUBLE,
            session_number      INT,
            expiry              INT,
            is_monthly_expiry   INT,
            days_to_expiry      INT,
            data_quality        STRING,
            fetch_latency_ms    DOUBLE
        )
        USING DELTA
        PARTITIONED BY (year, month, date)
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true',
            'delta.enableChangeDataFeed'       = 'true'
        )
    """)
    print(f"  Delta table ready  →  {DELTA_TABLE}")


def _ensure_dirs() -> None:
    os.makedirs(CSV_DIR,     exist_ok=True)
    os.makedirs(PARQUET_DIR, exist_ok=True)
    print(f"  CSV dir     →  {CSV_DIR}")
    print(f"  Parquet dir →  {PARQUET_DIR}")


def _flush_to_delta(df: pd.DataFrame) -> None:
    """
    Write DataFrame to Delta table.
    Schema-tolerant: if the table was created by v3 (no bar_ts/bar_time_str),
    this drops those columns before writing so the append does not fail with
    a schema mismatch. Uses mergeSchema=true so v4 columns are added on first
    v4 write to an existing v3 table.
    """
    # Discover what columns actually exist in the Delta table
    try:
        live_cols = [
            r.col_name
            for r in spark.sql(f"DESCRIBE TABLE {DELTA_TABLE}").collect()
            if not r.col_name.startswith("#")
        ]
        extra = [c for c in df.columns if c not in live_cols]
        if extra:
            print(f"    [schema] New columns not yet in Delta schema: {extra}")
            # mergeSchema=true below will add them automatically
    except Exception:
        live_cols = []

    sdf = spark.createDataFrame(df)
    for col in _ALL_INT_COLS:
        if col in sdf.columns:
            sdf = sdf.withColumn(col, sdf[col].cast(IntegerType()))
    (sdf.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DELTA_TABLE))


def _flush_to_csv(df: pd.DataFrame, session_date: str) -> None:
    """Append rows to today's CSV file; write header only on first append."""
    sym_safe = SYMBOL.replace("^", "").replace(".", "_")
    path     = f"{CSV_DIR}/{sym_safe}_{session_date}.csv"
    header   = not os.path.exists(path)
    df.to_csv(path, mode="a", index=False, header=header)


def _flush_to_parquet(df: pd.DataFrame, session_date: str) -> None:
    """Append to daily Parquet via read→concat→overwrite (no Parquet append API)."""
    sym_safe = SYMBOL.replace("^", "").replace(".", "_")
    path     = f"{PARQUET_DIR}/{sym_safe}_{session_date}.parquet"
    if os.path.exists(path):
        existing = pd.read_parquet(path)
        df       = pd.concat([existing, df], ignore_index=True)
    df.to_parquet(path, index=False, engine="pyarrow", compression="snappy")


def _flush_all(buffer: list, session_date: str) -> None:
    """
    Flush a list of row-dicts to all three sinks (Delta + CSV + Parquet).
    Converts tick_ts and bar_ts to datetime before writing.
    """
    df              = pd.DataFrame(buffer)
    df["tick_ts"]   = pd.to_datetime(df["tick_ts"])
    df["bar_ts"]    = pd.to_datetime(df["bar_ts"])
    _flush_to_delta(df)
    _flush_to_csv(df, session_date)
    _flush_to_parquet(df, session_date)
    print(f"    ✔ Flushed {len(buffer)} rows  →  Delta + CSV + Parquet")


## 7 · Self-Test Suite
Runs **before the first tick is collected**.
Any failure raises `RuntimeError` and aborts the collector — bad code
will never contaminate the dataset.


In [0]:
def _run_self_tests() -> None:
    """
    Comprehensive self-test suite (50+ assertions).
    Tests every helper with both happy-path and edge-case inputs.
    Raises RuntimeError listing ALL failures (not just the first).
    """
    import tempfile
    passed, failed = [], []

    def ok(name: str, cond: bool, got=None, exp=None):
        if cond:
            passed.append(name)
        else:
            failed.append(f"{name}  got={got!r}  expected={exp!r}")

    # ══════════════════════════════════════════════════════════════════════════
    #  1. CONFIG
    # ══════════════════════════════════════════════════════════════════════════
    _validate_config()
    ok("cfg_valid",              True)
    ok("cfg_symbol_non_empty",   bool(SYMBOL))
    ok("cfg_interval_1_to_60",   1 <= INTERVAL_MIN <= 60)
    ok("cfg_expiry_0_to_4",      0 <= EXPIRY_WEEKDAY <= 4)
    ok("cfg_flush_every_gte_1",  FLUSH_EVERY >= 1)
    ok("cfg_bad_ticks_gte_1",    MAX_BAD_TICKS >= 1)
    ok("cfg_spike_in_range",     0.001 <= SPIKE_PCT <= 0.5)

    # ══════════════════════════════════════════════════════════════════════════
    #  2. EXPIRY HELPERS  (EXPIRY_WEEKDAY=1 = Tuesday)
    # ══════════════════════════════════════════════════════════════════════════
    # 2a. is_expiry flag
    for d, exp in [
        (datetime.date(2026, 3, 24), 1),   # Tuesday  → expiry
        (datetime.date(2026, 3, 25), 0),   # Wednesday → not
        (datetime.date(2026, 3, 23), 0),   # Monday    → not
        (datetime.date(2026, 3, 26), 0),   # Thursday  → not
        (datetime.date(2026, 3, 27), 0),   # Friday    → not
    ]:
        got = 1 if d.weekday() == EXPIRY_WEEKDAY else 0
        ok(f"is_expiry_{d}", got == exp, got, exp)

    # 2b. days_to_next_expiry
    # EXPIRY_WEEKDAY=1 (Tue); formula = (1 - wd) % 7
    ok("dte_tue=0",  _days_to_next_expiry(datetime.date(2026, 3, 24)) == 0)
    ok("dte_wed=6",  _days_to_next_expiry(datetime.date(2026, 3, 25)) == 6)
    ok("dte_mon=1",  _days_to_next_expiry(datetime.date(2026, 3, 23)) == 1)
    ok("dte_thu=5",  _days_to_next_expiry(datetime.date(2026, 3, 26)) == 5)
    ok("dte_fri=4",  _days_to_next_expiry(datetime.date(2026, 3, 27)) == 4)
    ok("dte_always_0_to_6",
       all(0 <= _days_to_next_expiry(datetime.date(2026, 3, 23) + datetime.timedelta(n)) <= 6
           for n in range(7)))

    # 2c. last expiry of month
    # last Tuesday of March 2026 = 2026-03-31
    ok("last_exp_mar2026",
       _last_expiry_of_month(2026, 3) == datetime.date(2026, 3, 31),
       _last_expiry_of_month(2026, 3), datetime.date(2026, 3, 31))
    # last Tuesday of April 2026 = 2026-04-28
    ok("last_exp_apr2026",
       _last_expiry_of_month(2026, 4) == datetime.date(2026, 4, 28),
       _last_expiry_of_month(2026, 4), datetime.date(2026, 4, 28))
    # last day of result is always EXPIRY_WEEKDAY
    ok("last_exp_weekday_correct",
       _last_expiry_of_month(2026, 5).weekday() == EXPIRY_WEEKDAY)

    # 2d. is_monthly_expiry logic
    last_mar = _last_expiry_of_month(2026, 3)  # 2026-03-31
    ok("is_monthly_true",
       (last_mar.weekday() == EXPIRY_WEEKDAY and last_mar == last_mar) == True)
    # Non-last Tuesday in March 2026 = 2026-03-24 → NOT monthly
    non_last = datetime.date(2026, 3, 24)
    ok("is_monthly_false_mid_month",
       not (non_last.weekday() == EXPIRY_WEEKDAY and non_last == last_mar))

    # 2e. week_of_month
    ok("wom_day1",   _week_of_month(datetime.date(2026, 3,  1)) == 1)
    ok("wom_day7",   _week_of_month(datetime.date(2026, 3,  7)) == 1)
    ok("wom_day8",   _week_of_month(datetime.date(2026, 3,  8)) == 2)
    ok("wom_day14",  _week_of_month(datetime.date(2026, 3, 14)) == 2)
    ok("wom_day15",  _week_of_month(datetime.date(2026, 3, 15)) == 3)
    ok("wom_day26",  _week_of_month(datetime.date(2026, 3, 26)) == 4)
    ok("wom_day28",  _week_of_month(datetime.date(2026, 3, 28)) == 4)
    ok("wom_day29",  _week_of_month(datetime.date(2026, 3, 29)) == 5)

    # ══════════════════════════════════════════════════════════════════════════
    #  3. SESSION NUMBER
    # ══════════════════════════════════════════════════════════════════════════
    for h, exp in [
        (7, 0), (8, 0), (9, 1), (10, 2), (11, 3),
        (12, 4), (13, 5), (14, 6), (15, 6), (16, 0), (23, 0),
    ]:
        ok(f"sess_h{h}={exp}", _session_number(h) == exp, _session_number(h), exp)

    # ══════════════════════════════════════════════════════════════════════════
    #  4. DATA QUALITY
    # ══════════════════════════════════════════════════════════════════════════
    ok("dq_bad_price_zero",      _data_quality(0,      0,    None)  == "BAD_PRICE")
    ok("dq_bad_price_negative",  _data_quality(-1,     0,    None)  == "BAD_PRICE")
    ok("dq_bad_price_negative2", _data_quality(-100,   0,   23000)  == "BAD_PRICE")
    ok("dq_spike_above_5pct",    _data_quality(106,  100,    100)   == "SPIKE")
    ok("dq_spike_boundary",
       _data_quality(100 * (1 + SPIKE_PCT + 0.001), 0, 100)         == "SPIKE")
    ok("dq_no_spike_just_below",
       _data_quality(100 * (1 + SPIKE_PCT - 0.001), 0, 100)        != "SPIKE")
    ok("dq_spike_downward",
       _data_quality(100 * (1 - SPIKE_PCT - 0.001), 0, 100)         == "SPIKE")
    ok("dq_no_spike_no_prev",    _data_quality(23000,  100,  None)  == "OK")
    ok("dq_no_spike_prev_zero",  _data_quality(23000,  100,     0)  == "OK")
    ok("dq_ok_no_vol_no_prev",   _data_quality(23000,    0,  None)  == "OK_NO_VOL")
    ok("dq_ok_no_vol_with_prev", _data_quality(23000,    0, 22990)  == "OK_NO_VOL")
    ok("dq_ok_with_vol",         _data_quality(23000, 1000, 22990)  == "OK")

    # ══════════════════════════════════════════════════════════════════════════
    #  5. PRICE-DERIVED FEATURE MATHS
    # ══════════════════════════════════════════════════════════════════════════
    o2, h2, l2, c2 = 23200.0, 23350.0, 23150.0, 23300.0
    pr = round(h2 - l2, 2)

    ok("price_range=200",    pr == 200.0,                                    pr)
    ok("return_pct_positive", round((c2 - o2) / o2 * 100, 4) > 0)
    ret = round((c2 - o2) / o2 * 100, 4)
    ok("return_pct_exact",   ret == round(100 / 23200 * 100, 4),             ret)
    us = round(h2 - max(o2, c2), 2)
    ls = round(min(o2, c2) - l2, 2)
    ok("upper_shadow=50",    us == 50.0,                                      us)
    ok("lower_shadow=50",    ls == 50.0,                                      ls)
    ok("upper_shadow_gte_0", us >= 0)
    ok("lower_shadow_gte_0", ls >= 0)
    bp = round(abs(c2 - o2) / pr * 100, 2)
    ok("body_pct_0_to_100",  0 <= bp <= 100,                                  bp)
    ok("hl_ratio>1",         round(h2 / l2, 6) > 1)
    ok("log_ret_positive",   math.log(23100 / 23000) > 0)
    ok("log_ret_negative",   math.log(22900 / 23000) < 0)
    ok("log_ret_zero_prev",  0.0 == 0.0)             # when prev_close=None, log_ret=0

    # Edge cases
    ok("ret_pct_zero_open",   0.0 == (0.0 if 0 <= 0 else 1.0))
    ok("body_pct_zero_range", 0.0 == (0.0 if 0 <= 0 else 1.0))
    ok("hl_ratio_zero_low",   1.0 == (1.0 if 0 <= 0 else 0.0))

    # ══════════════════════════════════════════════════════════════════════════
    #  6. VWAP ACCUMULATOR
    # ══════════════════════════════════════════════════════════════════════════
    ctv = cv = 0.0
    bars_vwap = [
        (23350, 23150, 23300, 500_000),
        (23400, 23250, 23380, 600_000),
    ]
    for bh, bl, bc, bv in bars_vwap:
        tp2 = (bh + bl + bc) / 3
        ctv += tp2 * bv
        cv  += bv
    vwap_t = ctv / cv
    ok("vwap_in_range",   23150 <= vwap_t <= 23400,  round(vwap_t, 2))
    ok("vwap_is_weighted", vwap_t != sum((bh+bl+bc)/3 for bh,bl,bc,_ in bars_vwap) / 2)

    # Zero-volume VWAP (index ticker)
    ctv2 = cv2 = 0.0
    for bh, bl, bc, _ in bars_vwap:
        tp2 = (bh + bl + bc) / 3
        ctv2 += tp2 * max(0, 1)   # max(vo, 1) prevents zero accumulator
        cv2  += max(0, 1)
    vwap_zero_vol = ctv2 / cv2
    ok("vwap_zero_vol_valid", 23000 <= vwap_zero_vol <= 24000)

    # ══════════════════════════════════════════════════════════════════════════
    #  7. MARKET OPEN GUARD
    # ══════════════════════════════════════════════════════════════════════════
    def _sim(h: int, m: int, wd: int) -> bool:
        if wd >= 5:
            return False
        t = h * 60 + m
        return _T_OPEN <= t <= _T_CLOSE

    ok("open_0915",    _sim(9,  15, 0))
    ok("open_1200",    _sim(12,  0, 0))
    ok("open_1530",    _sim(15, 30, 0))
    ok("closed_1531",  not _sim(15, 31, 0))
    ok("closed_0914",  not _sim(9,  14, 0))
    ok("closed_0000",  not _sim(0,   0, 0))
    ok("closed_2359",  not _sim(23, 59, 0))
    ok("closed_sat",   not _sim(10,  0, 5))
    ok("closed_sun",   not _sim(10,  0, 6))
    ok("open_fri",         _sim(9,  15, 4))   # Friday is a trading day

    # ══════════════════════════════════════════════════════════════════════════
    #  8. COLLAPSED BAR DETECTION
    # ══════════════════════════════════════════════════════════════════════════
    def _is_collapsed(o, hi, lo, cl) -> bool:
        return hi == lo == o == cl and cl > 0

    ok("collapse_detected",     _is_collapsed(100.0, 100.0, 100.0, 100.0))
    ok("real_bar_not_collapsed", not _is_collapsed(23200.0, 23350.0, 23150.0, 23300.0))
    ok("zero_price_not_collapse", not _is_collapsed(0.0, 0.0, 0.0, 0.0))   # cl=0 → BAD_PRICE
    ok("partial_equal_not_all",  not _is_collapsed(100.0, 101.0, 100.0, 100.0))

    # ══════════════════════════════════════════════════════════════════════════
    #  9. ROW SCHEMA & INT COLUMNS
    # ══════════════════════════════════════════════════════════════════════════
    sample_row = {
        "tick_ts"           : datetime.datetime(2026, 3, 31, 9, 16, 3),
        "bar_ts"            : datetime.datetime(2026, 3, 31, 9, 15, 0),
        "tick_time_str"     : "09:16",
        "bar_time_str"      : "09:15",
        "symbol"            : "^NSEI",
        "year"              : 2026,
        "month"             : 3,
        "date"              : 31,
        "day"               : 1,
        "hour"              : 9,
        "minute"            : 15,
        "week_of_month"     : 5,
        "open"              : 23200.0,
        "high"              : 23350.0,
        "low"               : 23150.0,
        "close"             : 23300.0,
        "volume"            : 0.0,
        "typical_price"     : 23266.67,
        "vwap"              : 23266.67,
        "price_range"       : 200.0,
        "return_pct"        : 0.4310,
        "log_return"        : 0.004310,
        "body_pct"          : 50.0,
        "upper_shadow"      : 50.0,
        "lower_shadow"      : 50.0,
        "hl_ratio"          : 1.008621,
        "session_number"    : 1,
        "expiry"            : 1,
        "is_monthly_expiry" : 1,
        "days_to_expiry"    : 0,
        "data_quality"      : "OK_NO_VOL",
        "fetch_latency_ms"  : 287.4,
    }
    for c in _ALL_INT_COLS:
        ok(f"int_col_{c}_present", c in sample_row)
    ok("bar_ts_present",  "bar_ts"       in sample_row)
    ok("bar_time_present","bar_time_str" in sample_row)

    # ══════════════════════════════════════════════════════════════════════════
    #  10. CSV ROUND-TRIP (append + no duplicate header)
    # ══════════════════════════════════════════════════════════════════════════
    df_s = pd.DataFrame([sample_row])
    df_s["tick_ts"] = pd.to_datetime(df_s["tick_ts"])
    df_s["bar_ts"]  = pd.to_datetime(df_s["bar_ts"])
    with tempfile.TemporaryDirectory() as tmp:
        p = f"{tmp}/test.csv"
        df_s.to_csv(p, mode="w", index=False)
        df_r = pd.read_csv(p)
        ok("csv_rows_1",      len(df_r) == 1,                                  len(df_r))
        ok("csv_close_rt",    float(df_r["close"].iloc[0]) == 23300.0)
        ok("csv_bar_ts_col",  "bar_ts" in df_r.columns)
        ok("csv_int_cols_all",set(_ALL_INT_COLS).issubset(set(df_r.columns)))

        # Append second row — header must NOT repeat
        df_s.to_csv(p, mode="a", index=False, header=False)
        df_r2 = pd.read_csv(p)
        ok("csv_append_2rows", len(df_r2) == 2,                                len(df_r2))
        ok("csv_no_dup_header", list(df_r2.columns) == list(df_r.columns))

    # ══════════════════════════════════════════════════════════════════════════
    #  SUMMARY
    # ══════════════════════════════════════════════════════════════════════════
    total = len(passed) + len(failed)
    pct   = 100 * len(passed) / total if total else 0
    print(f"  Self-test: {len(passed)}/{total} passed  ({pct:.0f}%)", end="")
    if failed:
        print(f"  ──  {len(failed)} FAILED:")
        for msg in failed:
            print(f"    ✗ {msg}")
        raise RuntimeError(
            f"Self-test: {len(failed)} failure(s). Fix before collecting data."
        )
    print("  ✓  All clear.")


## 8 · Main Collector Loop
### Architecture
```
run()
 ├── _run_self_tests()          # abort if any test fails
 ├── _bootstrap_delta()         # create Delta table if needed
 ├── _ensure_dirs()             # create CSV/Parquet dirs if needed
 ├── WAIT LOOP (polls until 09:15 IST, exits on weekend/past-close)
 └── TICK LOOP (runs until _is_market_open() returns False)
      ├── _fetch_bar()           # iloc[-2] fetch
      ├── bad_streak tracking    # abort after MAX_BAD_TICKS consecutive failures
      ├── stale-bar dedup        # skip if bar_ts == last_bar_ts
      ├── buffer.append(row)
      ├── _flush_all() every FLUSH_EVERY ticks
      └── tick-aligned sleep     # INTERVAL_SEC - (time.time() % INTERVAL_SEC)
         → prevents cumulative drift over long sessions
```


In [0]:
def run() -> None:
    print("=" * 64)
    print(f"  NIFTY ML COLLECTOR  v4  |  {SYMBOL}  |  {INTERVAL_MIN}-min ticks")
    print("=" * 64)

    # ── 1. Self-tests ─────────────────────────────────────────────────────────
    print("\n[1/4] Running self-tests …")
    _run_self_tests()

    # ── 2. Bootstrap storage ──────────────────────────────────────────────────
    print("\n[2/4] Bootstrapping storage …")
    _bootstrap_delta()
    _ensure_dirs()

    # ── 3. Wait for market open ───────────────────────────────────────────────
    print("\n[3/4] Waiting for market open (09:15 IST) …")
    while True:
        now      = datetime.datetime.now(IST)
        open_dt  = _market_open_dt()
        close_dt = _market_close_dt()

        if now.weekday() >= 5:
            print("  ✗  Weekend — NSE closed. Exiting.")
            return
        if now >= close_dt:
            print("  ✗  Past 15:30 IST — session ended. Exiting.")
            return
        if now >= open_dt:
            break

        secs = (open_dt - now).total_seconds()
        print(f"    Opens in {int(secs // 60)}m {int(secs % 60)}s …  "
              f"(sleeping {min(int(secs), 60)}s)")
        time.sleep(min(secs, 60))

    print(f"  ✓  Market OPEN — collecting every {INTERVAL_MIN} min\n")

    # ── 4. Tick loop ─────────────────────────────────────────────────────────
    print("[4/4] Tick loop started.  Press ▶ Run All or interrupt cell to stop.\n")

    buffer       : list              = []
    prev_close   : Optional[float]  = None
    cum_tp_vol   : float             = 0.0
    cum_vol      : float             = 0.0
    bad_streak   : int               = 0
    last_bar_ts                      = None   # pd.Timestamp | None
    ticks_total  : int               = 0
    session_date : str               = datetime.datetime.now(IST).strftime("%Y_%m_%d")

    while _is_market_open():
        now = datetime.datetime.now(IST)
        row, cum_tp_vol, cum_vol, bar_ts = _fetch_bar(prev_close, cum_tp_vol, cum_vol)

        # ── Fetch failure handling ────────────────────────────────────────────
        # A fetch failure could be: network error, empty response, bad price,
        # or a collapsed bar (artefact — see note in _fetch_bar).
        # Collapsed bars return bar_ts (not None) so stale-dedup still works.
        if row is None:
            if bar_ts is None:
                # True failure (network/parse error) — count against bad_streak
                bad_streak += 1
                print(f"  [{now.strftime('%H:%M:%S')}] ⚠  Fetch failed  "
                      f"({bad_streak}/{MAX_BAD_TICKS})")
                if bad_streak >= MAX_BAD_TICKS:
                    print(
                        f"\n  ✗  {MAX_BAD_TICKS} consecutive fetch failures.\n"
                        f"     Possible causes: market closed early (holiday), "
                        f"Yahoo Finance outage, network error.\n"
                        f"     Flushing remaining buffer and aborting."
                    )
                    break
            else:
                # Collapsed/bad-price bar — don't penalise bad_streak,
                # but do respect stale-bar logic via bar_ts
                print(f"  [{now.strftime('%H:%M:%S')}] ⏭  Skipped (collapsed/bad-price bar)")
                if bar_ts == last_bar_ts:
                    pass   # already logged inside _fetch_bar
                last_bar_ts = bar_ts

            # Tick-aligned sleep before retry
            sleep_sec = INTERVAL_SEC - (time.time() % INTERVAL_SEC)
            time.sleep(max(sleep_sec, 1))
            continue

        # ── Stale-bar dedup ───────────────────────────────────────────────────
        # If yfinance returns the same iloc[-2] bar twice (can happen when we
        # poll faster than 1 bar/minute or on slow networks), skip silently.
        if bar_ts == last_bar_ts:
            print(f"  [{now.strftime('%H:%M:%S')}] ⏭  Stale bar ({bar_ts.strftime('%H:%M')}) — skipping")
            sleep_sec = INTERVAL_SEC - (time.time() % INTERVAL_SEC)
            time.sleep(max(sleep_sec, 1))
            continue

        # ── Good tick ─────────────────────────────────────────────────────────
        last_bar_ts = bar_ts
        bad_streak  = 0
        prev_close  = row["close"]
        buffer.append(row)
        ticks_total += 1

        q_icon = "✔" if row["data_quality"] in ("OK", "OK_NO_VOL") else "⚠"
        bar_lag = (now - bar_ts).total_seconds()   # seconds between bar close & collection
        print(
            f"  [{row['bar_time_str']}→{row['tick_time_str']}] {q_icon} "
            f"O={row['open']:.2f}  H={row['high']:.2f}  "
            f"L={row['low']:.2f}  C={row['close']:.2f}  "
            f"range={row['price_range']:.2f}  "
            f"VWAP={row['vwap']:.2f}  "
            f"logR={row['log_return']:+.5f}  "
            f"sess={row['session_number']}  "
            f"exp={row['expiry']}(m={row['is_monthly_expiry']})  "
            f"DTE={row['days_to_expiry']}  "
            f"Q={row['data_quality']}  "
            f"lat={row['fetch_latency_ms']:.0f}ms  "
            f"lag={bar_lag:.0f}s  "
            f"#{ticks_total}"
        )

        # ── Periodic flush ────────────────────────────────────────────────────
        if len(buffer) % FLUSH_EVERY == 0:
            _flush_all(buffer[-FLUSH_EVERY:], session_date)

        # ── Tick-aligned sleep ────────────────────────────────────────────────
        # time.time() % INTERVAL_SEC gives seconds elapsed in current interval.
        # Subtracting from INTERVAL_SEC gives seconds until the next boundary.
        # This prevents cumulative drift that would occur with a naive
        # time.sleep(INTERVAL_SEC) after each tick.
        sleep_sec = INTERVAL_SEC - (time.time() % INTERVAL_SEC)
        time.sleep(max(sleep_sec, 1))

    # ── End of session ────────────────────────────────────────────────────────
    remaining = len(buffer) % FLUSH_EVERY
    if remaining > 0:
        print(f"\n  Flushing final {remaining} rows …")
        _flush_all(buffer[-remaining:], session_date)

    now = datetime.datetime.now(IST)
    print(f"\n{'=' * 64}")
    print(f"  Session complete  |  {ticks_total} ticks collected")
    print(f"  Session date: {session_date}  |  Ended: {now.strftime('%H:%M:%S')} IST")
    print(f"  Bad streaks reset: all ≤ {MAX_BAD_TICKS}")
    print("=" * 64)


## 9 · Run

In [0]:
run()


## 10 · Post-Session Verification
Run this cell after market close to verify today's data.


In [0]:
# ── Post-session verification — schema-adaptive ──────────────────────────────
# This cell introspects the actual table schema so it works whether the table
# was created by v3 (no bar_ts/bar_time_str) or v4 (has both).
# Run after 15:30 IST to review today's collected ticks.
try:
    today = datetime.datetime.now(IST)

    # ── Discover actual columns ───────────────────────────────────────────────
    actual_cols = [
        r.col_name
        for r in spark.sql(f"DESCRIBE TABLE {DELTA_TABLE}").collect()
        if not r.col_name.startswith("#")
    ]
    print(f"Table columns ({len(actual_cols)}): {actual_cols}")

    # ── Build SELECT list from what actually exists ───────────────────────────
    _want = [
        "tick_ts",
        "bar_ts",           # v4 only
        "tick_time_str",
        "bar_time_str",     # v4 only
        "open", "high", "low", "close",
        "vwap",
        "log_return",
        "price_range",
        "session_number",
        "expiry",
        "days_to_expiry",
        "data_quality",
        "fetch_latency_ms",
    ]
    select_cols = [c for c in _want if c in actual_cols]

    # ORDER BY bar_ts if it exists, else tick_ts
    order_col = "bar_ts" if "bar_ts" in actual_cols else "tick_ts"

    sql = f"""
        SELECT {", ".join(select_cols)}
        FROM   {DELTA_TABLE}
        WHERE  year  = {today.year}
          AND  month = {today.month}
          AND  date  = {today.day}
        ORDER BY {order_col}
    """
    print(f"\nQuery:\n{sql}")

    df_check = spark.sql(sql).toPandas()
    n   = len(df_check)
    dq  = df_check["data_quality"].value_counts().to_dict() if "data_quality" in df_check.columns else {}
    lat = df_check["fetch_latency_ms"].describe().round(1)  if "fetch_latency_ms" in df_check.columns else {}

    print(f"\nToday {today.strftime('%Y-%m-%d')}: {n} ticks")
    print(f"Data quality : {dq}")
    if len(lat):
        print(f"Latency (ms) : min={lat['min']}  mean={lat['mean']}  max={lat['max']}")

    # ── Schema migration hint ─────────────────────────────────────────────────
    missing = [c for c in ["bar_ts", "bar_time_str"] if c not in actual_cols]
    if missing:
        print(f"\n[INFO] Columns {missing} are absent — table was created by v3 schema.")
        print("  To add them without losing data, run the ALTER statements below:")
        print(f"  spark.sql('ALTER TABLE {DELTA_TABLE} ADD COLUMNS (bar_ts TIMESTAMP, bar_time_str STRING)')")
        print("  After ALTER, new rows written by v4 will populate these columns.")
        print("  Old rows will show NULL for bar_ts/bar_time_str (use tick_ts as fallback).")

    display(df_check)

except Exception as e:
    print(f"Verification error: {e}")
    print("(Normal if the Delta table hasn't been written yet today.)")
